# Day 2 — Evo2 embeddings + classifier (the "teacher")

**Evo2** is a large DNA foundation model (StripedHyena2 architecture) trained on genomes
across the tree of life. Instead of hand-designing features (k-mers) or learning them from
scratch on our small dataset (the CNN), we can ask a model that has already "read" a huge
amount of DNA to represent our windows for us — this is the same "embeddings as black box"
move ESM/ProtBERT make for proteins.

We access Evo2 through **NVIDIA's hosted NIM API** rather than running the 7B-parameter
model ourselves. The organizers already called the API on every labeled window from
`01_kmer_and_cnn_baselines.ipynb` and saved the results — this is the single biggest
feasibility lever for the week: you never wait on a live API call.

In [ ]:
import sys
sys.path.append("../src")

import torch
from embeddings import load_supervised_embeddings
from models.classifier_heads import MLPHead
from eval import evaluate_logits, count_params, measure_latency_torch
from viz import plot_embedding_space

EMB_DIR = "../../data/supervised/embeddings"

X_train, y_train, ids_train = load_supervised_embeddings(EMB_DIR, "train")
X_val, y_val, ids_val = load_supervised_embeddings(EMB_DIR, "val")
print("embedding dim:", X_train.shape[1], "| train windows:", X_train.shape[0])

## Visualize the embedding space before training anything

In [ ]:
plot_embedding_space(X_train, y_train, method="pca",
                      title="Evo2 embeddings, colored by coding/non-coding label")

## Train the teacher: a small MLP head on top of frozen embeddings

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)

teacher = MLPHead(d_in=X_train.shape[1])
optimizer = torch.optim.Adam(teacher.parameters(), lr=1e-3)

n = X_train_t.shape[0]
for epoch in range(20):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        logits = teacher(X_train_t[idx])
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y_train_t[idx])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch+1}: loss={epoch_loss/n:.4f}")

In [ ]:
with torch.no_grad():
    val_logits = teacher(X_val_t).numpy()
teacher_metrics = evaluate_logits(val_logits, y_val)
print("Evo2 embeddings + MLP teacher:", teacher_metrics)
print("params:", count_params(teacher), "| latency (ms/sample):",
      measure_latency_torch(teacher, X_val_t[:1]))

### Checkpoint

Compare `teacher_metrics` against Day 1's `comparison` table — the Evo2 embeddings should
give a visible accuracy jump, and the PCA plot should already show some cluster separation
by label even though the embeddings were never fine-tuned for this task.

Save `teacher`, `X_train_t`, `y_train_t` (or re-load them) for the next notebook — the
teacher's predictions become the soft targets for distillation.

Next: `03_knowledge_distillation.ipynb`.